# Classical Supervised Learning on the Forest Cover Type Dataset

`covtype.csv` has 581,012 forest cells (30m × 30m) described by 10 numeric terrain features, 4 wilderness-area flags and 40 soil-type flags. The label `Cover_Type` is one of 7 tree species classes.

**Regression** (target: `Elevation`, a continuous variable)
1. Linear regression
2. Polynomial regression
3. Ridge and Lasso

**Classification** (target: `Cover_Type`)
4. Logistic regression
5. K-nearest neighbours

Both parts use random subsamples so every cell runs in reasonable time.

In [ ]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, RidgeCV, LassoCV, LogisticRegression, lasso_path
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, classification_report, confusion_matrix
)

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (6, 4)

## 0. Load and explore

In [ ]:
df = pd.read_csv("covtype.csv")
numeric_cols = df.columns[:10].tolist()
binary_cols = df.columns[10:-1].tolist()
TARGET = "Cover_Type"

print(df.shape)
print("missing values:", df.isna().sum().sum())
print("duplicate rows:", df.duplicated().sum())
df[numeric_cols].describe().T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = df[TARGET].value_counts().sort_index()
axes[0].bar(counts.index.astype(str), counts.values)
axes[0].set_title("Class distribution")
axes[0].set_xlabel("Cover_Type")
sns.boxplot(data=df.sample(50000, random_state=SEED), x=TARGET, y="Elevation", ax=axes[1])
axes[1].set_title("Elevation by cover type")
plt.tight_layout()
plt.show()

print((counts / counts.sum()).round(4))

In [ ]:
sample = df.sample(50000, random_state=SEED)
plt.figure(figsize=(8, 6))
sns.heatmap(sample[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 7})
plt.title("Correlation of numeric features")
plt.show()

# Part 1: Regression

The task is to predict `Elevation` from the remaining terrain, wilderness and soil features. `Cover_Type` is left out so the model uses only physical measurements.

In [ ]:
reg_target = "Elevation"
reg_num = [c for c in numeric_cols if c != reg_target]
reg_df = df.sample(50000, random_state=SEED)

X_reg = reg_df[reg_num + binary_cols]
y_reg = reg_df[reg_target]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=SEED)
print(Xr_train.shape, Xr_test.shape)
print("target mean/std:", y_reg.mean().round(1), y_reg.std().round(1))


def reg_metrics(model, name):
    pred_tr, pred_te = model.predict(Xr_train), model.predict(Xr_test)
    return {
        "model": name,
        "train_RMSE": np.sqrt(mean_squared_error(yr_train, pred_tr)),
        "test_RMSE": np.sqrt(mean_squared_error(yr_test, pred_te)),
        "test_MAE": mean_absolute_error(yr_test, pred_te),
        "train_R2": r2_score(yr_train, pred_tr),
        "test_R2": r2_score(yr_test, pred_te),
    }


baseline_rmse = np.sqrt(mean_squared_error(yr_test, np.full(len(yr_test), yr_train.mean())))
print("predict-the-mean RMSE:", round(baseline_rmse, 1))

## 1.1 Linear regression

Ordinary least squares fits $\hat y = w^\top x + b$ by minimising the sum of squared residuals. Features are standardised so coefficients are comparable.

In [ ]:
def make_preprocessor(degree=1):
    numeric = (
        make_pipeline(StandardScaler(), PolynomialFeatures(degree, include_bias=False), StandardScaler())
        if degree > 1 else StandardScaler()
    )
    return ColumnTransformer([("num", numeric, reg_num), ("bin", "passthrough", binary_cols)])


linear = Pipeline([("prep", make_preprocessor(1)), ("model", LinearRegression())])
linear.fit(Xr_train, yr_train)

results = [reg_metrics(linear, "Linear")]
pd.DataFrame(results).round(3)

In [ ]:
pred = linear.predict(Xr_test)
resid = yr_test - pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(yr_test, pred, s=3, alpha=0.3)
lims = [yr_test.min(), yr_test.max()]
axes[0].plot(lims, lims, "r--")
axes[0].set_xlabel("actual elevation")
axes[0].set_ylabel("predicted")
axes[0].set_title("Predicted vs actual")
axes[1].scatter(pred, resid, s=3, alpha=0.3)
axes[1].axhline(0, color="r", linestyle="--")
axes[1].set_xlabel("predicted")
axes[1].set_ylabel("residual")
axes[1].set_title("Residuals")
axes[2].hist(resid, bins=50)
axes[2].set_title("Residual distribution")
plt.tight_layout()
plt.show()

In [ ]:
names = reg_num + binary_cols
coefs = pd.Series(linear.named_steps["model"].coef_, index=names)
top = coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(15)
top.sort_values().plot.barh(figsize=(6, 5))
plt.title("Largest linear coefficients")
plt.show()

## 1.2 Polynomial regression

Adding powers and interactions of the numeric features lets a linear model fit curves. The model stays linear in its parameters, but the feature count grows quickly with the degree.

In [ ]:
rng = np.random.default_rng(SEED)
feature = "Horizontal_Distance_To_Roadways"
idx = rng.choice(len(reg_df), 60, replace=False)
small = reg_df.iloc[idx]
rest = reg_df.drop(reg_df.index[idx]).sample(3000, random_state=SEED)

grid = np.linspace(reg_df[feature].min(), reg_df[feature].max(), 400).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, degree in zip(axes, [1, 3, 12]):
    m = make_pipeline(StandardScaler(), PolynomialFeatures(degree, include_bias=False), StandardScaler(), LinearRegression())
    m.fit(small[[feature]], small[reg_target])
    ax.scatter(rest[feature], rest[reg_target], s=2, alpha=0.2, color="gray")
    ax.scatter(small[feature], small[reg_target], s=20, color="tab:blue")
    ax.plot(grid, m.predict(pd.DataFrame(grid, columns=[feature])), color="red")
    te = np.sqrt(mean_squared_error(rest[reg_target], m.predict(rest[[feature]])))
    tr = np.sqrt(mean_squared_error(small[reg_target], m.predict(small[[feature]])))
    ax.set_title(f"degree {degree} | train RMSE {tr:.0f} | test RMSE {te:.0f}")
    ax.set_xlabel(feature)
axes[0].set_ylabel("Elevation")
plt.tight_layout()
plt.show()

With only 60 training points, a degree-12 curve chases noise: training error falls while error on unseen data rises. This is overfitting in its simplest form.

In [ ]:
poly_models, poly_results = {}, []
for degree in [1, 2, 3, 4]:
    t0 = time.time()
    m = Pipeline([("prep", make_preprocessor(degree)), ("model", LinearRegression())])
    m.fit(Xr_train, yr_train)
    poly_models[degree] = m
    row = reg_metrics(m, f"Poly degree {degree}")
    row["n_features"] = m.named_steps["prep"].transform(Xr_train.iloc[:5]).shape[1]
    row["fit_seconds"] = time.time() - t0
    poly_results.append(row)

pd.DataFrame(poly_results).round(3)

In [ ]:
pr = pd.DataFrame(poly_results)
fig, ax = plt.subplots()
ax.plot(pr["model"].str[-1], pr["train_RMSE"], "o-", label="train")
ax.plot(pr["model"].str[-1], pr["test_RMSE"], "o-", label="test")
ax.set_xlabel("degree")
ax.set_ylabel("RMSE")
ax.set_title("Polynomial degree vs error (all numeric features)")
ax.legend()
plt.show()

## 1.3 Ridge and Lasso

Both add a penalty on coefficient size to the squared error.

Ridge: $\sum (y-\hat y)^2 + \alpha\sum w_j^2$ shrinks all coefficients smoothly.
Lasso: $\sum (y-\hat y)^2 + \alpha\sum |w_j|$ can set coefficients exactly to zero, so it also selects features.

First, coefficient paths on the standardised linear features.

In [ ]:
scaler = StandardScaler().fit(Xr_train[reg_num])
Z = scaler.transform(Xr_train[reg_num])
alphas = np.logspace(-2, 4, 40)

ridge_path = np.array([Ridge(alpha=a).fit(Z, yr_train).coef_ for a in alphas])
alphas_l, lasso_coefs, _ = lasso_path(Z, yr_train.values, alphas=np.logspace(-2, 2.5, 40))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for j, name in enumerate(reg_num):
    axes[0].plot(alphas, ridge_path[:, j], label=name)
    axes[1].plot(alphas_l, lasso_coefs[j], label=name)
axes[0].set_xscale("log")
axes[1].set_xscale("log")
axes[0].set_title("Ridge coefficient path")
axes[1].set_title("Lasso coefficient path")
for ax in axes:
    ax.set_xlabel("alpha")
    ax.set_ylabel("coefficient")
axes[1].legend(fontsize=7, loc="upper right")
plt.tight_layout()
plt.show()

Ridge coefficients shrink towards zero but stay nonzero. Lasso coefficients hit zero one by one as the penalty grows.

Regularization matters most when there are many correlated features, so next it is applied to the degree-3 polynomial model.

In [ ]:
prep3 = make_preprocessor(3)
Xp_train = prep3.fit_transform(Xr_train)
Xp_test = prep3.transform(Xr_test)
print("degree-3 feature count:", Xp_train.shape[1])

alpha_grid = np.logspace(-3, 3, 13)
rows = []
for a in alpha_grid:
    r = Ridge(alpha=a).fit(Xp_train, yr_train)
    rows.append({"alpha": a,
                 "ridge_train": np.sqrt(mean_squared_error(yr_train, r.predict(Xp_train))),
                 "ridge_test": np.sqrt(mean_squared_error(yr_test, r.predict(Xp_test)))})
ridge_sweep = pd.DataFrame(rows)

lasso_alphas = np.logspace(-1, 1.7, 7)
rows = []
for a in lasso_alphas:
    l = Lasso(alpha=a, max_iter=2000, tol=1e-3).fit(Xp_train, yr_train)
    rows.append({"alpha": a,
                 "lasso_train": np.sqrt(mean_squared_error(yr_train, l.predict(Xp_train))),
                 "lasso_test": np.sqrt(mean_squared_error(yr_test, l.predict(Xp_test))),
                 "nonzero": int((l.coef_ != 0).sum())})
lasso_sweep = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].semilogx(ridge_sweep["alpha"], ridge_sweep["ridge_train"], "o-", label="train")
axes[0].semilogx(ridge_sweep["alpha"], ridge_sweep["ridge_test"], "o-", label="test")
axes[0].set_title("Ridge: RMSE vs alpha")
axes[1].semilogx(lasso_sweep["alpha"], lasso_sweep["lasso_train"], "o-", label="train")
axes[1].semilogx(lasso_sweep["alpha"], lasso_sweep["lasso_test"], "o-", label="test")
axes[1].set_title("Lasso: RMSE vs alpha")
axes[2].semilogx(lasso_sweep["alpha"], lasso_sweep["nonzero"], "o-")
axes[2].set_title("Lasso: nonzero coefficients")
for ax in axes:
    ax.set_xlabel("alpha")
axes[0].legend()
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 13), cv=5).fit(Xp_train, yr_train)
lasso_cv = LassoCV(alphas=np.logspace(-1, 1.7, 7), cv=3, max_iter=2000, tol=1e-3, n_jobs=-1).fit(Xp_train, yr_train)
print("best ridge alpha:", ridge_cv.alpha_)
print("best lasso alpha:", lasso_cv.alpha_, "| nonzero coefs:", int((lasso_cv.coef_ != 0).sum()), "of", Xp_train.shape[1])

In [ ]:
final = []
for name, model in [
    ("Linear (degree 1)", poly_models[1]),
    ("Polynomial degree 3, OLS", poly_models[3]),
    ("Polynomial degree 3 + Ridge", Pipeline([("prep", make_preprocessor(3)), ("model", Ridge(alpha=ridge_cv.alpha_))]).fit(Xr_train, yr_train)),
    ("Polynomial degree 3 + Lasso", Pipeline([("prep", make_preprocessor(3)), ("model", Lasso(alpha=lasso_cv.alpha_, max_iter=2000, tol=1e-3))]).fit(Xr_train, yr_train)),
]:
    final.append(reg_metrics(model, name))

reg_summary = pd.DataFrame(final).set_index("model").round(3)
reg_summary

Plain degree-3 least squares fits the training set better than the linear model but does worse on the test set, a sign of overfitting from 263 correlated features. Ridge and Lasso remove that gap and give the lowest test RMSE of the four models. Lasso does so while zeroing out some of the coefficients.

In [ ]:
lasso_coef = pd.Series(lasso_cv.coef_, index=prep3.get_feature_names_out())
kept = lasso_coef[lasso_coef != 0].abs().sort_values(ascending=False)
print("features kept by Lasso:", len(kept), "of", len(lasso_coef))
kept.head(15).sort_values().plot.barh(figsize=(7, 5))
plt.title("Largest surviving Lasso coefficients")
plt.show()

# Part 2: Classification

The task is to predict `Cover_Type` (7 classes) from all 54 features. Classes are imbalanced: types 1 and 2 make up about 85% of the data, so accuracy alone can hide poor performance on rare classes. Macro-F1 averages F1 over classes equally and is reported alongside accuracy.

In [ ]:
cls_df = df.sample(80000, random_state=SEED)
X_cls = cls_df[numeric_cols + binary_cols]
y_cls = cls_df[TARGET]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cls, y_cls, test_size=0.2, stratify=y_cls, random_state=SEED
)
print(Xc_train.shape, Xc_test.shape)
print(yc_train.value_counts(normalize=True).sort_index().round(4).to_dict())

cls_prep = ColumnTransformer([("num", StandardScaler(), numeric_cols), ("bin", "passthrough", binary_cols)])
class_labels = sorted(y_cls.unique())


def cls_metrics(model, name, X_te=None, y_te=None):
    X_te = Xc_test if X_te is None else X_te
    y_te = yc_test if y_te is None else y_te
    pred = model.predict(X_te)
    return {"model": name, "accuracy": accuracy_score(y_te, pred), "macro_F1": f1_score(y_te, pred, average="macro")}


def plot_confusion(model, title):
    cm = confusion_matrix(yc_test, model.predict(Xc_test), labels=class_labels, normalize="true")
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels)
    plt.xlabel("predicted")
    plt.ylabel("true")
    plt.title(title)
    plt.show()


dummy = DummyClassifier(strategy="most_frequent").fit(Xc_train, yc_train)
cls_results = [cls_metrics(dummy, "Majority class")]
pd.DataFrame(cls_results).round(4)

## 2.1 Logistic regression

Logistic regression models class probabilities with a linear score passed through a sigmoid (binary) or softmax (multiclass). It is a linear classifier, so decision boundaries are flat hyperplanes. The `C` parameter is the inverse regularization strength: small `C` means a stronger L2 penalty.

In [ ]:
logit = Pipeline([("prep", cls_prep), ("model", LogisticRegression(max_iter=1000))])
t0 = time.time()
logit.fit(Xc_train, yc_train)
logit_fit_time = time.time() - t0

cls_results.append(cls_metrics(logit, "Logistic regression"))
pd.DataFrame(cls_results).round(4)

In [ ]:
print(classification_report(yc_test, logit.predict(Xc_test), digits=3))
plot_confusion(logit, "Logistic regression: row-normalised confusion matrix")

In [ ]:
proba = logit.predict_proba(Xc_test.iloc[:5])
print(pd.DataFrame(proba, columns=class_labels).round(3))
print("predicted:", logit.predict(Xc_test.iloc[:5]))
print("actual:   ", yc_test.iloc[:5].values)

### Effect of regularization strength

In [ ]:
Xs_train, Xs_val, ys_train, ys_val = train_test_split(
    Xc_train, yc_train, test_size=0.25, stratify=yc_train, random_state=SEED
)

rows = []
for C in [1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100]:
    m = Pipeline([("prep", cls_prep), ("model", LogisticRegression(C=C, max_iter=1000))]).fit(Xs_train, ys_train)
    rows.append({"C": C,
                 "train_acc": accuracy_score(ys_train, m.predict(Xs_train)),
                 "val_acc": accuracy_score(ys_val, m.predict(Xs_val)),
                 "val_macro_F1": f1_score(ys_val, m.predict(Xs_val), average="macro")})
c_sweep = pd.DataFrame(rows)
print(c_sweep.round(4))

plt.semilogx(c_sweep["C"], c_sweep["train_acc"], "o-", label="train")
plt.semilogx(c_sweep["C"], c_sweep["val_acc"], "o-", label="validation")
plt.xlabel("C (inverse regularization)")
plt.ylabel("accuracy")
plt.legend()
plt.title("Logistic regression: C sweep")
plt.show()

### Handling class imbalance and non-linearity

`class_weight='balanced'` upweights rare classes. Adding polynomial features lets a linear classifier draw curved boundaries in the original feature space.

In [ ]:
balanced = Pipeline([("prep", cls_prep), ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))]).fit(Xc_train, yc_train)

poly_prep = ColumnTransformer([
    ("num", make_pipeline(StandardScaler(), PolynomialFeatures(2, include_bias=False), StandardScaler()), numeric_cols),
    ("bin", "passthrough", binary_cols),
])
logit_poly = Pipeline([("prep", poly_prep), ("model", LogisticRegression(max_iter=2000))]).fit(Xc_train, yc_train)

extra = [
    cls_metrics(balanced, "Logistic (balanced weights)"),
    cls_metrics(logit_poly, "Logistic + degree-2 features"),
]
pd.DataFrame(cls_results + extra).round(4)

In [ ]:
plot_confusion(balanced, "Logistic regression with balanced class weights")

In [ ]:
coef = pd.DataFrame(logit.named_steps["model"].coef_, index=class_labels, columns=numeric_cols + binary_cols)
plt.figure(figsize=(9, 3.5))
sns.heatmap(coef[numeric_cols], annot=True, fmt=".1f", cmap="coolwarm", center=0, annot_kws={"size": 7})
plt.title("Logistic regression coefficients (standardised numeric features) by class")
plt.ylabel("class")
plt.show()

## 2.2 K-nearest neighbours

KNN stores the training data and classifies a point by a vote among its $k$ closest training points. It has no training step, makes no linearity assumption, and is entirely driven by the distance metric, so how features are scaled changes which points count as close. Raw features with large ranges (elevation, distances to roads and fire points, in the thousands) dominate raw distances, while standardising gives every feature, including the 44 binary flags, comparable weight.

In [ ]:
knn = Pipeline([("prep", cls_prep), ("model", KNeighborsClassifier(n_neighbors=5))])
t0 = time.time()
knn.fit(Xc_train, yc_train)
knn_fit_time = time.time() - t0
t0 = time.time()
knn_pred = knn.predict(Xc_test)
knn_pred_time = time.time() - t0

print(f"fit: {knn_fit_time:.2f}s | predict: {knn_pred_time:.2f}s")
cls_results.append(cls_metrics(knn, "KNN (k=5, scaled)"))
pd.DataFrame(cls_results).round(4)

In [ ]:
unscaled = KNeighborsClassifier(n_neighbors=5).fit(Xc_train, yc_train)
scaled_acc = accuracy_score(yc_test, knn_pred)
unscaled_acc = accuracy_score(yc_test, unscaled.predict(Xc_test))
print(f"raw-feature KNN accuracy:    {unscaled_acc:.4f}")
print(f"scaled-feature KNN accuracy: {scaled_acc:.4f}")

Scaling is not automatically better. Here the raw version can win because the high-range features (above all elevation) are also the most informative, so letting them dominate the distance acts as an implicit feature weighting. Standardising treats weak features such as aspect and hillshade equally, which adds noise to the distances. This is checked properly below by tuning scaling together with the other KNN settings.

### Choosing k

In [ ]:
ks = [1, 3, 5, 9, 15, 25, 51]
probe = np.random.default_rng(SEED).choice(len(Xs_train), 4000, replace=False)
rows = []
for k in ks:
    m = Pipeline([("prep", cls_prep), ("model", KNeighborsClassifier(n_neighbors=k))]).fit(Xs_train, ys_train)
    rows.append({"k": k,
                 "train_acc": accuracy_score(ys_train.iloc[probe], m.predict(Xs_train.iloc[probe])),
                 "val_acc": accuracy_score(ys_val, m.predict(Xs_val)),
                 "val_macro_F1": f1_score(ys_val, m.predict(Xs_val), average="macro")})
k_sweep = pd.DataFrame(rows)
print(k_sweep.round(4))

plt.plot(k_sweep["k"], k_sweep["train_acc"], "o-", label="train")
plt.plot(k_sweep["k"], k_sweep["val_acc"], "o-", label="validation")
plt.xscale("log")
plt.xlabel("k")
plt.ylabel("accuracy")
plt.legend()
plt.title("KNN: accuracy vs k")
plt.show()

Training accuracy is 1.0 at $k=1$ because each point is its own nearest neighbour, yet $k=1$ also has the best validation accuracy, and accuracy and macro-F1 fall steadily as $k$ grows. Neighbouring forest cells in this dataset are very similar, so the single closest example is the most informative, and larger $k$ blurs rare classes into the majority ones. Note that this sweep uses standardised features; the next cell re-tests scaling together with the other settings at this $k$.

In [ ]:
best_k = int(k_sweep.loc[k_sweep["val_acc"].idxmax(), "k"])
print("best k on validation accuracy:", best_k)

preps = {"scaled": cls_prep, "raw": "passthrough"}
variants = {}
for prep_name, prep in preps.items():
    for weights in ["uniform", "distance"]:
        for metric in ["euclidean", "manhattan"]:
            m = Pipeline([("prep", prep),
                          ("model", KNeighborsClassifier(n_neighbors=best_k, weights=weights, metric=metric))]).fit(Xs_train, ys_train)
            variants[(prep_name, weights, metric)] = accuracy_score(ys_val, m.predict(Xs_val))
pd.Series(variants, name="val_acc").round(4)

In [ ]:
best_prep, best_weights, best_metric = max(variants, key=variants.get)
print("best variant:", best_prep, best_weights, best_metric)

knn_final = Pipeline([("prep", preps[best_prep]),
                      ("model", KNeighborsClassifier(n_neighbors=best_k, weights=best_weights, metric=best_metric))])
knn_final.fit(Xc_train, yc_train)

print(classification_report(yc_test, knn_final.predict(Xc_test), digits=3))
plot_confusion(knn_final, "KNN: row-normalised confusion matrix")

## 2.3 Logistic regression vs KNN

In [ ]:
t0 = time.time()
knn_final.predict(Xc_test)
knn_final_pred_time = time.time() - t0
t0 = time.time()
logit.predict(Xc_test)
logit_pred_time = time.time() - t0

compare = pd.DataFrame([
    {**cls_metrics(logit, "Logistic regression"), "fit_s": logit_fit_time, "predict_s": logit_pred_time},
    {**cls_metrics(logit_poly, "Logistic + degree-2"), "fit_s": np.nan, "predict_s": np.nan},
    {**cls_metrics(knn_final, f"KNN (k={best_k}, {best_prep}, {best_weights}, {best_metric})"), "fit_s": knn_fit_time, "predict_s": knn_final_pred_time},
]).set_index("model")
compare.round(4)

In [ ]:
sizes = [1000, 2500, 5000, 10000, 20000, 40000, len(Xc_train)]
rows = []
for n in sizes:
    Xn, _, yn, _ = train_test_split(Xc_train, yc_train, train_size=n, stratify=yc_train, random_state=SEED) if n < len(Xc_train) else (Xc_train, None, yc_train, None)
    for name, model in [
        ("Logistic", Pipeline([("prep", cls_prep), ("model", LogisticRegression(max_iter=1000))])),
        ("KNN", Pipeline([("prep", preps[best_prep]), ("model", KNeighborsClassifier(n_neighbors=best_k, weights=best_weights, metric=best_metric))])),
    ]:
        model.fit(Xn, yn)
        rows.append({"n_train": n, "model": name, "test_acc": accuracy_score(yc_test, model.predict(Xc_test))})

lc = pd.DataFrame(rows)
sns.lineplot(data=lc, x="n_train", y="test_acc", hue="model", marker="o")
plt.xscale("log")
plt.title("Test accuracy vs training set size")
plt.show()

The plot compares how each model's test accuracy changes with the amount of training data. Read it as: a linear model's accuracy is limited by its flat decision boundaries, while KNN depends directly on how densely the training data covers the feature space.

## Summary

In [ ]:
summary = pd.DataFrame({
    "task": ["regression"] * len(reg_summary) + ["classification"] * len(compare),
    "model": list(reg_summary.index) + list(compare.index),
    "primary_metric": [f"test RMSE {v:.1f}" for v in reg_summary["test_RMSE"]]
                      + [f"test acc {v:.3f}" for v in compare["accuracy"]],
    "secondary_metric": [f"test R2 {v:.3f}" for v in reg_summary["test_R2"]]
                        + [f"macro F1 {v:.3f}" for v in compare["macro_F1"]],
})
summary